In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_json_path = 'rca100/cases/t024/task.json'
with open(task_json_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information from the task.json
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "inventory错误次数突增" (inventory error count surge). It occurred between 2026-04-04 22:37:09 and 2026-04-04 22:45:01 (UTC+8). The entity that triggered the alert is an APM operation named `inventory::/api/v1/inventory/{productId}`. The user is requesting help to analyze the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('', {'start': '2026-04-04T22:37:09.273091+08:00', 'end': '2026-04-04T22:45:01+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">inventory错误次数突增<alert_event event_id="f117cea2bc1c5c47816e78c6fd7573c2" trans_id="sfhu5c5e6t7sfeo75j5he606o5" rule_id="663b6c72-c01c-49e2-b1a7-805ae65925b3" rule_name="inventory错误次数突增" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_path = 'rca100/cases/t024/metrics.parquet'
metrics_df = pd.read_parquet(metrics_path)

# List all available KPIs and their associated entity sets
kpi_entity_sets = metrics_df[['metric', 'entity_set']].drop_duplicates()

kpi_entity_sets

```
Out[2]:
```
The `metrics.parquet` file contains 63 unique KPIs across various entity sets. Some examples include `node_disk_usage_rate` and `node_memory_usage_rate` under `k8s.node`, as well as `error_rate` under both `apm.operation` and `apm.instance`. These KPIs can be used for anomaly detection and fault identification. For a comprehensive view, the full list of KPIs can be examined further.

The original code execution output of IPython Kernel is also provided below for reference:

metric          entity_set
0       node_not_ready_status            k8s.node
2100        node_ready_status            k8s.node
4200     node_disk_usage_rate            k8s.node
4760   node_memory_usage_rate            k8s.node
6300   node_pod_running_count            k8s.node
...                       ...                 ...
78274                workload  apm.service.legacy
78280                   error  apm.service.legacy
78283                    slow  apm.service.legacy
88491              error_rate       apm.operation
88575              error_rate        apm.instance

[63 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Convert the time column to numeric for filtering and calculations
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Filter metrics for the inventory service and other possible root cause components
inventory_metrics = metrics_df[
    (metrics_df['entity_set'].isin(['apm.operation', 'apm.instance'])) &
    (metrics_df['entity_name'].str.contains('inventory', case=False))
]

# Calculate global thresholds (P95) for each KPI across the entire dataset
global_thresholds = inventory_metrics.groupby(['metric'])['value'].quantile(0.95).reset_index()
global_thresholds.columns = ['metric', 'p95_threshold']

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-04-04T22:37:09.273091+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-04T22:45:01+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert alert time window to microseconds for filtering
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter data within the alert time window
filtered_metrics = inventory_metrics[
    (inventory_metrics['time'] >= alert_start_us) &
    (inventory_metrics['time'] <= alert_end_us)
]

global_thresholds, filtered_metrics

```
Out[3]:
```
The global thresholds (P95) for key KPIs of the inventory service are as follows: 
- `avg_request_latency_seconds`: 7.44 seconds
- `error_count`: 44
- `error_rate`: 0.41
- `request_count`: 5794
- `slow_count`: 227.5

Within the alert time window (2026-04-04 22:37:09 to 2026-04-04 22:45:01 UTC+8), 275 metric records were filtered for the inventory service. These records include KPIs such as `error_count` and `error_rate`, which are critical for identifying anomalies. Further analysis can focus on these filtered metrics to pinpoint the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(                        metric  p95_threshold
0  avg_request_latency_seconds       7.436849
1                  error_count      44.000000
2                   error_rate       0.407173
3                request_count    5794.000000
4                   slow_count     227.500000,                    time domain     entity_set                         entity_id                               entity_name       metric      value                            metric_set_id service
56988  1775313729000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  error_count  43.000000  apm@metric_set@apm.metric.apm.operation    None
56989  1775313759000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  error_count  43.000000  apm@metric_set@apm.metric.apm.operation    None
56990  1775313819000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  error_count   7.000000  apm@metric_set@apm.metric.apm.operation    None
56991  1775313849000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  error_count   7.000000  apm@metric_set@apm.metric.apm.operation    None
57068  1775313609000000    apm  apm.operation  0730d69c0818717e0d0e987385f3a3ab            inventory::/actuator/health/**  error_count   1.000000  apm@metric_set@apm.metric.apm.operation    None
...                 ...    ...            ...                               ...                                       ...          ...        ...                                      ...     ...
88678  1775313639000000    apm   apm.instance  7fb113e159d1fa7ccdfc641acf1870ad                inventory-794d9f8f85-hhjtj   error_rate   0.003663   apm@metric_set@apm.metric.apm.instance    None
88679  1775313819000000    apm   apm.instance  7fb113e159d1fa7ccdfc641acf1870ad                inventory-794d9f8f85-hhjtj   error_rate   0.032110   apm@metric_set@apm.metric.apm.instance    None
88680  1775313849000000    apm   apm.instance  7fb113e159d1fa7ccdfc641acf1870ad                inventory-794d9f8f85-hhjtj   error_rate   0.111111   apm@metric_set@apm.metric.apm.instance    None
88723  1775313729000000    apm   apm.instance  2da48118270a22f4de7ac642102def69                 inventory-887d8db89-7v4s5   error_rate   0.125356   apm@metric_set@apm.metric.apm.instance    None
88724  1775313759000000    apm   apm.instance  2da48118270a22f4de7ac642102def69                 inventory-887d8db89-7v4s5   error_rate   0.245810   apm@metric_set@apm.metric.apm.instance    None

[275 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Perform anomaly detection on the filtered metrics data
anomalies = filtered_metrics[
    ((filtered_metrics['metric'] == 'error_count') & (filtered_metrics['value'] > 44)) |
    ((filtered_metrics['metric'] == 'error_rate') & (filtered_metrics['value'] > 0.41))
]

# Select relevant columns for anomaly reporting
anomalies = anomalies[['time', 'metric', 'value']]

anomalies

```
Out[4]:
```
An anomaly was detected within the alert time window. At timestamp `1775313759000000` (2026-04-04 22:42:39 UTC+8), the `error_rate` exceeded the threshold of 0.41, reaching a value of 0.544. This indicates a significant issue with the inventory service during this period.

The original code execution output of IPython Kernel is also provided below for reference:

time      metric     value
88492  1775313759000000  error_rate  0.544304```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_path = 'rca100/cases/t024/traces.parquet'
traces_df = pd.read_parquet(traces_path)

# Convert relevant columns to numeric for filtering
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')

# Filter traces related to the inventory service within the alert time window
inventory_traces = traces_df[
    (traces_df['spanName'].str.contains('/api/v1/inventory/{productId}', case=False)) &
    (traces_df['startTime'] >= alert_start_us * 1000) &  # Convert microseconds to nanoseconds
    (traces_df['endTime'] <= alert_end_us * 1000)
]

# Identify the call chain by extracting parent-child relationships
call_chain = inventory_traces[['traceId', 'spanId', 'parentSpanId', 'spanName', 'serviceName', 'statusCode']]

call_chain

```
Out[5]:
```
The filtered traces reveal 1,574 spans related to the `inventory::/api/v1/inventory/{productId}` operation within the alert time window. All spans have a `statusCode` of 0, indicating no errors at the trace level for the inventory service itself. To determine if any upstream services are experiencing issues, further analysis of parent spans (`parentSpanId`) and their corresponding services is required.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId                       spanName serviceName statusCode
401572  6ac952524f35600f1a15022bb5d601da  711550d1d97eb60e  406ae2a65cfa3bde  /api/v1/inventory/{productId}   inventory          0
401573  621595cadf31c7ee57c21dc8d1e51877  a3e0ef7061e4ac39  a32f95f1e0b92596  /api/v1/inventory/{productId}   inventory          0
401579  1ffaf7753a843811bb93b77f0a4c7fb4  a4abaa74b9b94993  039e7083c9ba16ac  /api/v1/inventory/{productId}   inventory          0
401581  3e55c25d83673f1365d78e01cdab3752  3b5a8a84fc7e068a  95820e38f94aa1c6  /api/v1/inventory/{productId}   inventory          0
401583  9844d2e6a624303f69dd10c212727b17  4fa5e144c848a353  04326cfc2fd23df6  /api/v1/inventory/{productId}   inventory          0
...                                  ...               ...               ...                            ...         ...        ...
587160  b5cc24d60cb1c50695c71c353eb5c2ee  83064858bedafe50  279fcce18d8c7a0d  /api/v1/inventory/{productId}   inventory          0
599887  31b0194c13e7df644907914bf1baf506  caf46a003096f066  e680064bfed88e32  /api/v1/inventory/{productId}   inventory          0
599888  ac3e7d41c78d354dc75ad41114044e4a  872faf1cac34947b  da02e9cf8f16ca81  /api/v1/inventory/{productId}   inventory          0
599889  602c9db1ff58b185bf34d4b3089c513a  5b4f29f23eb0ee4d  f34308c5a1840fec  /api/v1/inventory/{productId}   inventory          0
599890  c222ff42fcd0fa78da6f097d3f7e81d7  014f32ece73cda44  2867da1e2868ab42  /api/v1/inventory/{productId}   inventory          0

[1574 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Filter parent spans from the traces data
parent_span_ids = inventory_traces['parentSpanId'].dropna().unique()

# Identify spans corresponding to the parentSpanIds
parent_spans = traces_df[traces_df['spanId'].isin(parent_span_ids)]

# Check for anomalies or errors in the parent spans
upstream_anomalies = parent_spans[parent_spans['statusCode'] != 0]

# Select relevant columns for reporting
upstream_anomalies = upstream_anomalies[['traceId', 'spanId', 'parentSpanId', 'spanName', 'serviceName', 'statusCode']]

upstream_anomalies

```
Out[6]:
```
The analysis of parent spans for the inventory service traces reveals 1,537 spans related to the upstream `cart` service. All spans have a `statusCode` of 0, indicating no errors or anomalies in the `cart` service during the alert time window. This suggests that the upstream `cart` service is functioning normally and is not contributing to the issue observed in the inventory service.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId spanName serviceName statusCode
401489  67060e0458611da242ee2ee173d9e3ce  ae6e520de5ac30c6  f820ac318ed20189      GET        cart          0
401490  0601375f08c0534d1fb63ada4781e8c9  aadeb4c2faffd61b  695ecf55fc63ab65      GET        cart          0
401491  ec9af00e80cc1933b534c7d3a29dda47  2f74b7720535c9c4  9d4293a5b12dcfc6      GET        cart          0
401492  c3c15f4b91e48d9130d8d1f28b757171  0260e94afe1d0d53  34b703c880e78bec      GET        cart          0
401493  362ac51ea8d9aeb16921448e0dd5ff18  fca5fb64e9e85a3c  d2e2fe21ea4a42f9      GET        cart          0
...                                  ...               ...               ...      ...         ...        ...
580173  602c9db1ff58b185bf34d4b3089c513a  f34308c5a1840fec  7c51266cc0da41a6      GET        cart          0
586194  b5cc24d60cb1c50695c71c353eb5c2ee  279fcce18d8c7a0d  6103674e9059eb6a      GET        cart          0
594773  c222ff42fcd0fa78da6f097d3f7e81d7  2867da1e2868ab42  68855675da48a4a3      GET        cart          0
594829  ac3e7d41c78d354dc75ad41114044e4a  da02e9cf8f16ca81  4288701ac98edb11      GET        cart          0
599211  31b0194c13e7df644907914bf1baf506  e680064bfed88e32  cb848ef3110834b0      GET        cart          0

[1537 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

# Load the logs.parquet file
logs_path = 'rca100/cases/t024/logs.parquet'
logs_df = pd.read_parquet(logs_path)

# Convert the `_time_` column to numeric for filtering
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs related to the inventory service within the alert time window
inventory_logs = logs_df[
    (logs_df['_container_name_'].str.contains('inventory', case=False, na=False)) &
    (logs_df['_time_'] >= alert_start_us) &
    (logs_df['_time_'] <= alert_end_us)
]

# Search for error messages or exceptions in the `content` field
error_logs = inventory_logs[inventory_logs['content'].str.contains('error|exception', case=False, na=False)]

# Select relevant columns for reporting
error_logs = error_logs[['_time_', 'content', '_container_name_', '_pod_name_']]

error_logs

```
Out[7]:
```
No error messages or exceptions were found in the logs related to the inventory service within the alert time window. This suggests that there are no explicit error logs indicating the root cause of the issue in the inventory service during this period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [_time_, content, _container_name_, _pod_name_]
Index: []```
```